# OTTO Round 08 · Worked feature examples
## Rarity-weighted historical-session evidence

All events below are invented unit fixtures. The 24 primary and 24 ablation columns are produced by the same pure feature implementation used in the real pipeline. Nothing in this notebook is an achieved model metric.

In [1]:
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
figures=[]
def emit(fig):
    fig.update_layout(margin=dict(l=65,r=35,t=80,b=65),height=480)
    figures.append(fig)
    display({'application/vnd.plotly.v1+json':json.loads(pio.to_json(fig))}, raw=True)

import numpy as np
from neighbors import HistoricalSession
from feature_logic import build, names, select
# Entirely synthetic historical sessions. No targets, models or Kaggle score.
def h(sid,aids,kinds,indices,timestamps):
    return HistoricalSession(sid,*[np.asarray(x,dtype=np.int64) for x in (aids,kinds,indices,timestamps)])
history={
 1:h(1,[9,10,80,90],[0,0,1,2],[0,1,2,4],[1000,2000,5000,15000]),
 2:h(2,[20,10,81,91],[0,0,1,2],[0,1,2,3],[1000,2000,5000,9000]),
 3:h(3,[10,20,82,92],[0,0,1,2],[0,1,2,7],[1000,2000,6000,12000]),
 4:h(4,[83,10,93],[1,0,2],[0,1,2],[1000,3000,4500]),
 5:h(5,[20,84,94],[0,1,2],[0,1,2],[1000,5000,10000]),
}
for session in history.values():session.validate(100000)
anchors=np.array([10,20],dtype=np.int64)
candidates=np.array([9,10,20,80,81,82,83,84,90,91,92,93,94,999],dtype=np.int64)
frequencies={10:100,20:5}
primary,ablation,diagnostics=build(8,anchors,candidates,list(history),history,frequencies)
primary_names=names(8); ablation_names=names(8,True)
assert primary.shape==ablation.shape==(len(candidates),24)
print('SYNTHETIC_FEATURE_EXAMPLE_ONLY:',diagnostics)

SYNTHETIC_FEATURE_EXAMPLE_ONLY: {'neighbor_counts': [5, 5], 'multi_anchor_counts': [2, 2], 'neighbor_identity_overlap': 5, 'primary_supported_pairs': 13, 'ablation_supported_pairs': 13}


## 1 · Clicks weighted support
Each source session counts once; unsupported candidates remain zero.

In [2]:
f=go.Figure()
f.add_trace(go.Bar(name='Primary',x=candidates.astype(str),y=primary[:,1]))
f.add_trace(go.Bar(name='Ablation',x=candidates.astype(str),y=ablation[:,1]))
f.update_layout(title='Synthetic clicks weighted-vote share',xaxis_title='Candidate product',yaxis_title='Descriptive share',barmode='group');emit(f)

## 2 · Carts weighted support
Each source session counts once; unsupported candidates remain zero.

In [3]:
f=go.Figure()
f.add_trace(go.Bar(name='Primary',x=candidates.astype(str),y=primary[:,9]))
f.add_trace(go.Bar(name='Ablation',x=candidates.astype(str),y=ablation[:,9]))
f.update_layout(title='Synthetic carts weighted-vote share',xaxis_title='Candidate product',yaxis_title='Descriptive share',barmode='group');emit(f)

## 3 · Orders weighted support
Each source session counts once; unsupported candidates remain zero.

In [4]:
f=go.Figure()
f.add_trace(go.Bar(name='Primary',x=candidates.astype(str),y=primary[:,17]))
f.add_trace(go.Bar(name='Ablation',x=candidates.astype(str),y=ablation[:,17]))
f.update_layout(title='Synthetic orders weighted-vote share',xaxis_title='Candidate product',yaxis_title='Descriptive share',barmode='group');emit(f)

## 4 · Where representations differ
This is a numerical comparison of toy features, not feature importance or ranking performance.

In [5]:
f=go.Figure(go.Bar(x=[n.split('_',2)[-1] for n in primary_names],y=np.abs(primary-ablation).mean(axis=0)))
f.update_layout(title='Synthetic mean absolute feature difference',yaxis_title='Mean absolute difference',xaxis_tickangle=-50,height=700);emit(f)
assert len(figures)==4
assert np.isfinite(primary).all() and np.isfinite(ablation).all()
print('WORKED_EXAMPLE_COMPLETE: 24 + 24 finite columns, no models')

WORKED_EXAMPLE_COMPLETE: 24 + 24 finite columns, no models


## Interpretation
Rarity weights can redistribute evidence even within the same source pool. Compare the strength and concentration of multi-anchor versus last-anchor histories; do not infer that rarity improves real validation from this fixture.